# Module 05 — t-Tests (Independent & Paired)
Dedicated notebook for Module 05 — reproduces all 3 t-test examples in the lesson: a significant independent test (teaching method), a non-significant independent test (gender), and a paired test (pretest/posttest).

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

SPSS/PSPP equivalents: `Analyze > Compare Means > Independent-Samples T Test` / `Paired-Samples T Test`.

In [ ]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset — SAME seed=2026 used
    throughout Modules 03-08, exactly matching lop_hoc_240.csv/.sav in the
    repo (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]

## 1. Independent t-test: teaching method → score gain — SIGNIFICANT
Step 1 is always Levene's Test: it decides whether to read the "Equal variances assumed" or "not assumed" row.

🎯 **What question does this method answer?** An independent-samples t-test tests: is the OBSERVED mean difference between two UNRELATED groups (not the same people) large enough to believe it reflects a REAL difference in the population, rather than just random sampling fluctuation? Null hypothesis (H0): the two groups have the same population mean.

In [ ]:
g1 = df.loc[df["method"]=="Project-based", "gain"]
g2 = df.loc[df["method"]=="Traditional", "gain"]
lev_stat, lev_p = stats.levene(g1, g2)
t, p = stats.ttest_ind(g1, g2, equal_var=(lev_p >= 0.05))
print(f"Levene: F={lev_stat:.3f}, p={lev_p:.3f} -> {'Equal variances assumed' if lev_p>=0.05 else 'not assumed'}")
print(f"Project-based: n={len(g1)}, M={g1.mean():.2f}, SD={g1.std(ddof=1):.2f}")
print(f"Traditional: n={len(g2)}, M={g2.mean():.2f}, SD={g2.std(ddof=1):.2f}")
print(f"t({len(g1)+len(g2)-2})={t:.3f}, p={p:.2e}")

#### 📤 Actual output
Levene p=0.535 (≥0.05 → read the "assumed" row). t(238)=7.589, p<0.001. Project-based M=8.66 SD=5.77 (n=115); Traditional M=2.89 SD=6.00 (n=125). ✅ Matches the lesson.

**APA-style write-up:** "The Project-based group's mean score gain (M=8.66, SD=5.77) was significantly higher (t(238)=7.59, p<0.001) than the Traditional group's (M=2.89, SD=6.00)."

## 2. Independent t-test: gender → Math score — NOT significant

In [ ]:
mA = df.loc[df["gender"]=="Male", "math"]
mB = df.loc[df["gender"]=="Female", "math"]
lev_stat2, lev_p2 = stats.levene(mA, mB)
t2, p2 = stats.ttest_ind(mA, mB, equal_var=(lev_p2 >= 0.05))
print(f"Levene: F={lev_stat2:.3f}, p={lev_p2:.3f}")
print(f"Male: n={len(mA)}, M={mA.mean():.2f}, SD={mA.std(ddof=1):.2f}")
print(f"Female: n={len(mB)}, M={mB.mean():.2f}, SD={mB.std(ddof=1):.2f}")
print(f"t({len(mA)+len(mB)-2})={t2:.3f}, p={p2:.3f}")

#### 📤 Actual output
Levene p=0.081 (≥0.05 but close to the threshold — worth caution). t(238)=-1.348, p=0.179 → **insufficient evidence** for a gender difference in Math scores in this sample. A "no difference found" result is also valuable and should be reported honestly, not "p-hacked" until significant.

## 3. Paired t-test: pretest → posttest (same 240 students)
Paired tests are more powerful than independent tests because they remove between-student variability — leaving only the variance attributable to the effect of learning.

🎯 **What question does this method answer?** A paired t-test tests: when the SAME group of subjects is measured twice (before/after, or two different variables), is the mean DIFFERENCE between the two measurements large enough to believe it reflects a REAL change (not just chance)? H0: the mean difference = 0.

In [ ]:
t3, p3 = stats.ttest_rel(df["posttest"], df["pretest"])
diff = df["posttest"] - df["pretest"]
print(f"Mean pretest={df['pretest'].mean():.2f} (SD={df['pretest'].std(ddof=1):.2f})")
print(f"Mean posttest={df['posttest'].mean():.2f} (SD={df['posttest'].std(ddof=1):.2f})")
print(f"Mean difference={diff.mean():.2f}")
print(f"t({len(df)-1})={t3:.2f}, p={p3:.2e}")
r_pp = stats.pearsonr(df["pretest"], df["posttest"])[0]
print(f"pretest-posttest correlation r={r_pp:.3f} (explains why paired is more powerful than independent)")

#### 📤 Actual output
Mean difference=5.65, t(239)=13.37, p<0.001. pretest-posttest correlation r≈0.857 — very high, exactly why pairing is much more powerful than independent for the same sample size. ✅ Matches the lesson.